# 🔌 The Universal Adapter
## One question. Three offices. No integration project.

**Section 10** of *Search, Extract & Value: The Lightning Fast Assistant* 🚗💨

---

| | |
|---|---|
| 🔑 **An API key** | `ANTHROPIC_API_KEY` in your `.env`. |
| 🌐 **Internet** | yes — the model runs on Anthropic's machines. |
| ⏱️ **Time** | about two minutes, and a few cents. |

> ### ☁️ This is the **hosted** version.
> The other version is **`10_universal_adapter_local.ipynb`**.
>
> 📌 **The three county servers still run on YOUR machine.** Only the model is
> remote. The databases never leave the building — the model is told the
> answers, not given the keys. Worth being precise about that with your IT
> department.

---

A taxpayer walks up to the counter and asks something completely ordinary:

> ### *"I want to split my lot and sell half. Can I?"*

Nobody in the building can answer that.

**The lot size** is on the Appraiser's roll. **Ownership** is with the Clerk of
Courts. **The tax balance** is with the Tax Collector. **The rule** is in a
policy manual on somebody's shelf. Three elected officials, three software
vendors, three databases that have never spoken to one another — and the only
thing joining them today is a person making three phone calls over a week.

This notebook connects all three to one assistant in about forty lines, and
**we do not write a single tool to do it.**

---

## What MCP actually is

Section 7 built a tool. We wrote a Python function, then hand-wrote a JSON
schema describing it, and handed both to the model. **Two pieces of code per
tool, every time, forever.** That works, and it is what most people mean when
they say "we integrated the AI".

The **Model Context Protocol** turns that round. An MCP server is a program
somebody else runs, which *publishes* a list of what it can do. Your agent
asks it "what have you got?" and it answers.

| | Section 7 | Section 10 |
|---|---|---|
| Who wrote the tool? | you did | the office that owns the data |
| Who wrote the schema? | you did, by hand | it describes itself |
| Adding a new source | new function + new schema | one line in a list |
| The office changes its system | your code breaks | their server changes, yours does not |

That is the difference between a charger built for one phone and a **USB-C
socket**. It is the Universal Adapter on the back of Mr. Fusion: you no longer
need a different engine for a different fuel.

---

## 1️⃣ Three programs, three offices

In [1]:
from delorean import agent, data, llm, prompts, ratio, viz
from delorean.offices import HILL_VALLEY, Offices

data.check()            # make sure Hill Valley County exists on this machine

# The model runs on Anthropic's machines. The three county servers below still
# run here. Only this cell differs from the local notebook.
client = llm.get_api_client()
respond = agent.api_responder(client, llm.api_model())

print(f"model ready: {llm.api_model()}")

model ready: claude-haiku-4-5


Now the part that matters.

`Offices` starts **three separate programs** — you can see them in your task
manager — and asks each one what it can do. Look carefully at what we pass it:
three file paths and a list of names. **No schemas. No function signatures. No
descriptions.**

In [2]:
# Re-running this cell? Close the previous servers first. Three orphaned
# subprocesses holding GPU memory is a real way to lose an afternoon.
try:
    offices.close()
except NameError:
    pass

offices = Offices(HILL_VALLEY, also_local=["search_policy"]).open()

for row in offices.catalogue():
    print(f"  {row['icon']} {row['office']:<17} {row['tool']:<20} "
          f"takes {row['takes']}")

print(f"\n{len(offices.catalogue())} tools, and we wrote none of them.")

  🏛️ Appraiser         get_parcel           takes ['parcel_id']
  🏛️ Appraiser         find_parcels         takes ['search']
  📜 Clerk of Courts   get_deeds            takes ['parcel_id']
  📜 Clerk of Courts   corrections_since    takes ['since_date']
  💰 Tax Collector     get_tax_status       takes ['parcel_id']

5 tools, and we wrote none of them.


Three programs started, five tools discovered, and **not one line of
this repository describes any of them.** Those names, those parameters and the
descriptions the model will read all came off the servers themselves.

Here is the same thing as the audience sees it. The dashed panel on the right
is our own policy manual — a plain local tool, not MCP — because the
difference between *"my filing cabinet"* and *"somebody else's system"* is the
distinction this whole section rests on.

In [3]:
viz.show_offices(offices.catalogue(), local=["search_policy"])

---

## 2️⃣ What we did *not* have to write

This is the act worth slowing down for.

In Section 7 a tool cost us a function **and** a hand-written schema. Here is
that schema, exactly as it appears in this repository — and then the same kind
of description for an MCP tool, which we did not write at all.

In [4]:
from delorean import tools as local_tools

# Section 7: written by hand, by us, and kept in step with the function by us.
ours = local_tools.schemas(["lookup_parcel"])[0]["function"]

# Section 10: this arrived down a pipe from a program in another office.
theirs = offices.schemas(["get_parcel"])[0]["function"]

for label, spec, who in [("SECTION 6  — we wrote this", ours, "us"),
                         ("SECTION 10 — the Appraiser's server wrote this",
                          theirs, "them")]:
    print("=" * 70)
    print(f"  {label}")
    print("=" * 70)
    print(f"  name       : {spec['name']}")
    print(f"  parameters : {list(spec['parameters'].get('properties', {}))}")
    print(f"  required   : {spec['parameters'].get('required', [])}")
    print(f"  description: {spec['description'].strip().splitlines()[0][:60]}...")
    print(f"  maintained by: {who}")
    print()

  SECTION 6  — we wrote this
  name       : lookup_parcel
  parameters : ['parcel_id']
  required   : ['parcel_id']
  description: Look up one parcel in the Hill Valley County assessment roll...
  maintained by: us

  SECTION 10 — the Appraiser's server wrote this
  name       : get_parcel
  parameters : ['parcel_id']
  required   : ['parcel_id']
  description: Look up one parcel on the Hill Valley County assessment roll...
  maintained by: them



**Both are the same shape**, because MCP hands back ordinary JSON
Schema and that is what the OpenAI tool format wraps anyway. Converting one to
the other is an envelope change, and it is nine lines in
[`offices.py`](../src/delorean/offices.py).

The difference is not technical. It is **who has to keep it correct**.

When the Clerk's office upgrades its system, the schema on the left is now a
lie and nobody finds out until an agent gives somebody a wrong answer. The
schema on the right changes at the same moment the system does, because it is
generated by the system.

> ### 🎯 The tool did not get easier to write. It stopped being ours to write.

---

## 3️⃣ *"May Biff Tannen split his lot?"* — Live Demo 7

The same graph as Section 9. The same `build_agent`. One argument different:

```python
agent.build_agent(respond, toolbox=offices)
```

`toolbox` is the whole of Section 10 in one line. Point it at the local
registry and the agent uses functions from this repository; point it at
`Offices` and it uses three programs in three other buildings. **Reason → Act →
Observe does not change.**

In [5]:
assistant = agent.build_agent(respond, max_steps=9, toolbox=offices)

QUESTION = ("Biff Tannen wants to split parcel 04-127-3891 into two lots. "
            "May he? Check everything the county requires.")

split = agent.ask(assistant, QUESTION, system=prompts.COUNTY_ASSISTANT)

for step in split["trace"]:
    if step["step"] == "act":
        owner = offices.owner_of(step["tool"])
        where = f"{owner.icon} {owner.label}" if owner else "📖 our policy manual"
        print(f"  {where:<24} {step['tool']}({step['args']})")

print()
print(agent.final_answer(split))

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

  🏛️ Appraiser             get_parcel({'parcel_id': '04-127-3891'})
  📜 Clerk of Courts        get_deeds({'parcel_id': '04-127-3891'})
  💰 Tax Collector          get_tax_status({'parcel_id': '04-127-3891'})
  📖 our policy manual      search_policy({'question': 'what are the requirements for a parcel split', 'top_k': 3})

**No, Biff Tannen may not split this parcel.** The lot size is adequate at 15,800 square feet, and Biff is the legal owner per the Clerk's deed recorded March 14, 2025, but the Tax Collector shows $8,432.13 in total outstanding taxes across two years. Policy 4.2 requires a zero tax balance before any split is approved—payment plans do not satisfy this requirement.


Watch which buildings it walked into.

In [6]:
viz.show_offices(offices.catalogue(), trace=split["trace"],
                 local=["search_policy"])

### What just happened

Four sources, one answer, and **every one of the three offices was consulted
even though the taxes alone were enough to refuse it.** That is deliberate, and
it is in the system prompt:

> *Check all three even after one of them fails. Somebody told only the first
> reason their application was refused has to come back a second time to hear
> the rest.*

That is not a demo trick. It is counter service. A taxpayer who is told *"you
owe back taxes"* and nothing else will pay the taxes, come back, and only then
learn about the ownership requirement.

⚠️ **Notice also what it did NOT do.** It did not decide the lot was big
enough — it asked the Appraiser and got 15,800 sq ft. It did not decide Biff
owns the parcel — it read the Clerk's deed and found Biff as grantee. It did
not estimate the balance — the Tax Collector said $8,432.13. **The model chose
the questions. The offices supplied the facts.**

> ### 🎯 Three phone calls and a week, in about twenty-five seconds.

---

## 4️⃣ *"Is our sales file still true?"* — Live Demo 8

Here is the part that should make you slightly uncomfortable.

The Appraiser's office does not work from the Clerk's live records. It works
from a **feed** — a file of recorded sales sent over periodically and loaded
into CAMA. Every ratio study in Section 9 was run against that file.

Feeds go stale. Worse, they go stale **silently**: a deed is recorded wrong,
somebody notices, the Clerk corrects it, and your copy never hears. Nobody is
careless. You simply have a photograph of a filing cabinet that has since been
rearranged.

In [7]:
pulled = data.feed_pulled_on()
print(f"our sales file was pulled from the Clerk on {pulled}\n")

sync = agent.ask(
    assistant,
    f"Our sales file was pulled from the Clerk on {pulled}. Has the Clerk "
    f"corrected any deeds since then that we would still be using?",
    system=prompts.COUNTY_ASSISTANT)

print(agent.final_answer(sync))

our sales file was pulled from the Clerk on 2026-01-15



Yes. The Clerk corrected one deed since your pull date: instrument 2025-000069 for parcel 12-393-7896 (George Rossi to Frank Yamada, recorded 2025-07-12). The deed type was changed to QUIT CLAIM and marked as not-arm's-length (arms_length = 0) on 2026-03-04. If your sales file includes this $647,000 transaction, you must remove it from any ratio study, since the price is no longer evidence of market value.


One deed. Now — does one deed matter?

It was booked as a **$647,000 arm's-length Warranty Deed**. The Clerk
reclassified it a **QUIT CLAIM**, which under IAAO standards means the price is
not evidence of market value and the sale must be excluded from a ratio study
altogether.

Section 9 studied it as a valid sale. Here is what that cost, and this is
arithmetic rather than opinion — the same `ratio` module Section 9 uses.

In [8]:
feed = data.sales_feed_df()
CORRECTED = "12-393-7896"
hood = feed[feed.parcel_id == CORRECTED].iloc[0]["neighborhood"]

stale = feed[feed.neighborhood == hood]
live = stale[stale.parcel_id != CORRECTED]

for label, sales in [(f"{hood} — our sales file", stale),
                     (f"{hood} — with the Clerk's correction", live)]:
    result = ratio.ratio_stats(sales)
    marks = "  ".join(
        f"{t['metric']} {t['value']:.4f} {t['verdict']}" for t in result["tests"])
    print(f"  {label}")
    print(f"    {marks}")
    print(f"    fails: {result['failed_tests'] or 'nothing'}")
    print()

  Courthouse Square — our sales file
    Median ratio 0.9262 PASS  COD 22.0789 FAIL  PRD 1.0278 PASS
    fails: ['Uniformity (COD)']

  Courthouse Square — with the Clerk's correction
    Median ratio 0.9146 PASS  COD 20.3075 FAIL  PRD 1.0419 FAIL
    fails: ['Uniformity (COD)', 'Equity (PRD)']



In [9]:
viz.show_scorecard(ratio.ratio_stats(live),
                   title=f"{hood} — with the Clerk's correction")

### Read that carefully

Removing a bad sale **did not make the neighborhood look better.**

With the stale file, Courthouse Square failed one test — uniformity. With the
Clerk's correction it fails **two**: uniformity *and* equity. That $647,000
quit-claim was a high-ratio outlier on an expensive property, and its presence
was propping the PRD up inside the standard.

> ### 🚨 The stale feed was not merely out of date. It was flattering us.

An office running that ratio study would have signed off on an equity result
that was not true, using a file it had every reason to trust, and nothing in
its own systems could have told it otherwise.

**This is the argument for MCP that costs money if you ignore it.** Not
convenience. Not fewer clicks. The difference between a number you can defend
at a hearing and one you cannot.

---

## 🧪 Explore on your own

Everything below is optional and off the live path.

### What a server actually looks like

The whole Tax Collector office is one short file. If you can read this, you can
publish your own office's data the same way.

In [10]:
from pathlib import Path

source = Path("../mcp_server/tax_server.py").read_text(encoding="utf-8")

# Split on a NEWLINE-prefixed decorator. The bare string also appears in the
# file's own header, where it is being talked ABOUT rather than used -- and
# splitting on that handed us the import block instead of the tool.
body = source.split(chr(10) + "@mcp.tool()")[1]
print("@mcp.tool()" + body[:1250])

@mcp.tool()
def get_tax_status(parcel_id: str) -> dict:
    """Look up the tax account for one parcel at the Tax Collector.

    Returns, in dollars: the levy for the year, how much has been paid, the
    balance still due, whether any prior year is also delinquent, and the
    total outstanding across all years.

    Use this whenever money owed on a parcel could matter — a split, a
    combination, a transfer, or a taxpayer asking what they owe. The
    Appraiser's roll and the Clerk's deeds do NOT carry this information.

    `total_outstanding` is the number that matters for eligibility questions:
    it includes the prior year balance, and a parcel with any outstanding
    amount at all is barred from a split under Policy 4.2.

    Args:
        parcel_id: The parcel number, written NN-NNN-NNNN, e.g. 04-127-3891.
    """
    canonical = normalize_parcel_id(parcel_id)
    if canonical is None:
        return {"ok": False,
                "error": f"{parcel_id!r} is not a Hill Valle

**The docstring is not documentation. It is the prompt.** That text is
what a language model reads when deciding whether to call the function — which
is why it names the units, says what the number means, and says explicitly that
the Appraiser and the Clerk do not hold this information.

A vague docstring here is indistinguishable from a broken tool.

### Ask your own question

In [11]:
your_question = "Who owns 1955 Hilldale Ct, and do they owe anything?"

answer = agent.ask(assistant, your_question,
                   system=prompts.COUNTY_ASSISTANT)

for step in answer["trace"]:
    if step["step"] == "act":
        owner = offices.owner_of(step["tool"])
        print(f"  asked {owner.label if owner else 'our policy manual'}")
print()
print(agent.final_answer(answer))

  asked Appraiser
  asked Clerk of Courts
  asked Tax Collector

**Biff Tannen** owns 1955 Hilldale Ct, according to the Clerk of Courts deed recorded March 14, 2025. Yes, he owes money—a total of **$8,432.13** outstanding across two tax years, with the 2025 tax year currently delinquent.


### Adding a fourth office

Nothing in `offices.py` knows what an Appraiser is. To add the Building
Department tomorrow you would write a server for them — or more likely, *they*
would — and add one line:

```python
Office("building", "Building Dept", "building_server.py",
       "permits and inspections", "🏗️")
```

No new tool. No new schema. No change to the agent, the graph, or this
notebook.

> ### 🎯 That is the claim. An office you have never heard of can become
> ### something your assistant can ask, without you writing code about it.

---

## 🧹 Shut the offices down

Three programs are running because this notebook started them. `Offices`
registers a cleanup handler so they go away when the kernel exits, but doing it
deliberately is a good habit — and after the hour this project lost to three
orphaned servers holding a graphics card, a habit worth having.

In [12]:
offices.close()
print("three offices closed")

three offices closed


---

## 🎬 What Section 10 was actually about

**Not a protocol.** The protocol is an implementation detail and it will be
replaced by something else eventually.

The thing that matters is **who is responsible for describing a system**. For
thirty years the answer has been *whoever wants to read it* — which is why
county integrations are expensive, brittle, and quietly out of date. MCP moves
that responsibility to the office that owns the data, where it was always
supposed to live.

| Section | What the agent could reach |
|---|---|
| 6 | one function we wrote |
| 8 | our own filing cabinet |
| 7 | our own database |
| **10** | **three systems we do not own and did not describe** |

And the payload, one more time:

> ### 💡 A taxpayer asked a question no single office could answer, and got a
> ### correct, sourced, citable answer in twenty-five seconds — from three
> ### systems that have never been integrated, on a machine under a desk.

**Section 11 is the take-home.** Everything you have watched today is in this
repository, and the next section is about what you do with it on Monday.